In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import os 

BASE_DIR = os.getcwd()

TRUE_DATA_PATHS = [
    os.path.join(BASE_DIR, "results/true/test1_true.csv"),
    os.path.join(BASE_DIR, "results/true/test2_true.csv"),
]

TEST1_S22U = [
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_looking_left.csv"),
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_looking_right.csv"),
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_swing_left.csv"),
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_swing_right.csv"),
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_calling_left.csv"),
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_calling_right.csv")
]

TEST1_S20P = [
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_looking_left.csv"),
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_looking_right.csv"),
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_swing_left.csv"),
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_swing_right.csv"),
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_calling_left.csv"),
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_calling_right.csv")
]

TEST2_S22U = [
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S22U_looking.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S22U_swing.csv"),
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S22U_calling.csv")
]

TEST2_S20P = [
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S20P_looking.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S20P_swing.csv"),
    # os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S20P_calling.csv")
]

CONV_PDR_PATHS = [
    # os.path.join(BASE_DIR, "results/conv/test1_looking_left.csv"),
    # os.path.join(BASE_DIR, "results/conv/test1_looking_right.csv"),
    # os.path.join(BASE_DIR, "results/conv/test1_swing_left.csv"),
    # os.path.join(BASE_DIR, "results/conv/test1_swing_right.csv"),
    # os.path.join(BASE_DIR, "results/conv/test1_calling_left.csv"),
    # os.path.join(BASE_DIR, "results/conv/test1_calling_right.csv"),
    # os.path.join(BASE_DIR, "results/conv/test2_looking.csv"),
    os.path.join(BASE_DIR, "results/conv/test2_swing.csv"),
    # os.path.join(BASE_DIR, "results/conv/test2_calling.csv")
]

In [ ]:
# ============================================================
# 2. 실험 조건 이름
# ============================================================

TEST1_CASES = [
    "Looking Left",
    "Looking Right",
    "Swing Left",
    "Swing Right",
    "Calling Left",
    "Calling Right",
]

TEST2_CASES = [
    "Looking",
    "Swing",
    "Calling",
]


# ============================================================
# 3. 유틸 함수
# ============================================================

def load_csv(path):
    """
    CSV 파일 로드 함수.
    파일 존재 여부를 먼저 확인하여 디버깅을 쉽게 함.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")
    return pd.read_csv(path)


def get_xy(df, preferred_x=None, preferred_y=None):
    """
    DataFrame에서 x, y 좌표 컬럼을 자동으로 찾는 함수.

    우선순위:
    - 사용자가 지정한 preferred_x, preferred_y
    - pred_X, pred_Y
    - x_m, y_m
    - X, Y
    - x, y
    """
    x_candidates = []
    y_candidates = []

    if preferred_x is not None:
        x_candidates.append(preferred_x)
    if preferred_y is not None:
        y_candidates.append(preferred_y)

    x_candidates += ["pred_X", "x_m", "X", "x", "pos_x", "Position_X"]
    y_candidates += ["pred_Y", "y_m", "Y", "y", "pos_y", "Position_Y"]

    x_col = None
    y_col = None

    for col in x_candidates:
        if col in df.columns:
            x_col = col
            break

    for col in y_candidates:
        if col in df.columns:
            y_col = col
            break

    if x_col is None or y_col is None:
        raise KeyError(
            "x, y 좌표 컬럼을 찾을 수 없습니다.\n"
            f"현재 컬럼: {list(df.columns)}"
        )

    return df[x_col].values, df[y_col].values


def apply_axis_flip(x, y, x_flag=False, y_flag=False):
    """
    좌표축 반전 함수.
    """
    x = np.asarray(x).copy()
    y = np.asarray(y).copy()

    if x_flag:
        x = -x
    if y_flag:
        y = -y

    return x, y


# ============================================================
# 4. Trajectory Plot 함수
# ============================================================

def plot_traj(
    true_df,
    s22u_df,
    s20p_df,
    conv_df=None,
    title=None,
    x_flag=False,
    y_flag=False,
    conv_x_flag=False,
    conv_y_flag=False,
    test_flag=1,
    save_path=None,
    show=True,
):
    """
    True Trajectory, Proposed S22U, Proposed S20+, Conventional PDR을 함께 시각화.

    x_flag, y_flag:
        Proposed Method 좌표축 반전 여부

    conv_x_flag, conv_y_flag:
        Conventional PDR 좌표축 반전 여부
    """

    # -----------------------------
    # 좌표 로드
    # -----------------------------
    true_x, true_y = get_xy(true_df, preferred_x="x_m", preferred_y="y_m")
    s22u_x, s22u_y = get_xy(s22u_df, preferred_x="pred_X", preferred_y="pred_Y")
    s20p_x, s20p_y = get_xy(s20p_df, preferred_x="pred_X", preferred_y="pred_Y")

    # Proposed Method만 축 반전
    s22u_x, s22u_y = apply_axis_flip(
        s22u_x,
        s22u_y,
        x_flag=x_flag,
        y_flag=y_flag,
    )

    s20p_x, s20p_y = apply_axis_flip(
        s20p_x,
        s20p_y,
        x_flag=x_flag,
        y_flag=y_flag,
    )

    # Conventional PDR은 별도 플래그 적용
    if conv_df is not None:
        conv_x, conv_y = get_xy(conv_df)
        conv_x, conv_y = apply_axis_flip(
            conv_x,
            conv_y,
            x_flag=conv_x_flag,
            y_flag=conv_y_flag,
        )
    else:
        conv_x, conv_y = None, None

    # -----------------------------
    # 축 범위 설정
    # -----------------------------
    if test_flag == 1:
        x_lim = (-20, 20)
        y_lim = (-30, 10)
    else:
        x_lim = (-30, 30)
        y_lim = (-25, 10)

    # -----------------------------
    # Plot
    # -----------------------------
    fig, ax = plt.subplots(figsize=(12, 8))

    ax.plot(
        true_x,
        true_y,
        label="Ground Truth",
        color="blue",
        linewidth=3,
    )

    if conv_df is not None:
        ax.plot(
            conv_x,
            conv_y,
            label="Conventional PDR",
            color="black",
            linestyle=":",
            linewidth=2.5,
        )

    ax.plot(
        s22u_x,
        s22u_y,
        label="Proposed Method (S22U)",
        color="orange",
        linestyle="--",
        linewidth=2.5,
    )

    ax.plot(
        s20p_x,
        s20p_y,
        label="Proposed Method (S20+)",
        color="green",
        linestyle="--",
        linewidth=2.5,
    )

    # -----------------------------
    # 시작 / 종료 지점 표시
    # -----------------------------
    ax.scatter(
        true_x[0],
        true_y[0],
        marker="s",
        color="red",
        s=120,
        label="Start & End Point",
        zorder=10,
    )

    ax.scatter(
        true_x[-1],
        true_y[-1],
        marker="*",
        color="blue",
        s=220,
        label="GT End Point",
        zorder=10,
    )

    ax.scatter(
        s22u_x[-1],
        s22u_y[-1],
        marker="^",
        color="orange",
        s=160,
        label="S22U End Point",
        zorder=10,
    )

    ax.scatter(
        s20p_x[-1],
        s20p_y[-1],
        marker="D",
        color="green",
        s=130,
        label="S20+ End Point",
        zorder=10,
    )

    if conv_df is not None:
        ax.scatter(
            conv_x[-1],
            conv_y[-1],
            marker="X",
            color="black",
            s=140,
            label="Conventional PDR End Point",
            zorder=10,
        )

    # -----------------------------
    # 축 / 스타일
    # -----------------------------
    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel("X (m)", fontsize=16)
    ax.set_ylabel("Y (m)", fontsize=16)

    if title is not None:
        ax.set_title(title, fontsize=18, pad=14)

    ax.grid(True, linestyle="--", alpha=0.5)

    ax.tick_params(axis="both", labelsize=13)

    # ax.legend(
    #     loc="upper center",
    #     bbox_to_anchor=(0.5, -0.10),
    #     ncol=2,
    #     fontsize=13,
    #     frameon=True,
    #     framealpha=0.95,
    # )

    # plt.tight_layout(rect=[0, 0.08, 1, 1])

    if save_path is not None:
        save_dir = os.path.dirname(save_path)
        if save_dir != "":
            os.makedirs(save_dir, exist_ok=True)

        plt.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
        )

    if show:
        plt.show()
    else:
        plt.close(fig)


# ============================================================
# 5. 데이터 로드
# ============================================================

#test1_true_df = load_csv(TRUE_DATA_PATHS[0])
test2_true_df = load_csv(TRUE_DATA_PATHS[1])

#test1_s22u_dfs = [load_csv(path) for path in TEST1_S22U]
#test1_s20p_dfs = [load_csv(path) for path in TEST1_S20P]

test2_s22u_dfs = [load_csv(path) for path in TEST2_S22U]
test2_s20p_dfs = [load_csv(path) for path in TEST2_S20P]

conv_dfs = [load_csv(path) for path in CONV_PDR_PATHS]

#test1_conv_dfs = conv_dfs[:6]
#test2_conv_dfs = conv_dfs[6:]


# ============================================================
# 6. Test 1 Plot
# ============================================================

SAVE_DIR = os.path.join(BASE_DIR, "results", "figures")
os.makedirs(SAVE_DIR, exist_ok=True)

# for idx, case_name in enumerate(TEST1_CASES):
#     plot_traj(
#         true_df=test1_true_df,
#         s22u_df=test1_s22u_dfs[idx],
#         s20p_df=test1_s20p_dfs[idx],
#         conv_df=test1_conv_dfs[idx],
#         #title=f"Test 1 - {case_name}",
#         x_flag=True,
#         y_flag=True,
#         test_flag=1,
#         save_path=os.path.join(
#             SAVE_DIR,
#             f"test1_{case_name.lower().replace(' ', '_')}.png"
#         ),
#         show=True,
#     )


# ============================================================
# 7. Test 2 Plot
# ============================================================

for idx, case_name in enumerate(TEST2_CASES):
    plot_traj(
        true_df=test2_true_df,
        s22u_df=test2_s22u_dfs[idx],
        s20p_df=test2_s20p_dfs[idx],
        conv_df=conv_dfs[idx],
        #title=f"Test 2 - {case_name}",
        x_flag=True,
        y_flag=True,
        conv_x_flag=True,
        conv_y_flag=True,
        test_flag=2,
        save_path=os.path.join(
            SAVE_DIR,
            f"test2_{case_name.lower().replace(' ', '_')}.png"
        ),
        show=True,
    )

IndexError: list index out of range